# BW \#102 WordPress
This week, we'll thus examine the Git repository used in WordPress development. We'll do this by looking at two CSV files I created based on the Git logs. The first file contains all of the commits in the trunk branch for the WordPress project. The second file contains the commits, along with the number of lines added to and/or removed from the project in each commit.

WordPress started as a system that let you create, edit, and manage a blog without needing too much technical knowledge: Install WordPress, or choose a hosting provider who will do that for you, and you're able to do everything via a Web interface.
Better yet, WordPress is $\textbf{open-source}$ software, meaning that anyone can download, use, install, modify, or distribute it free of charge. A recent report (https://www.wpbeginner.com/research/ultimate-list-of-wordpress-stats-facts-and-other-research/) says that WordPress powers more than 40 percent of Web sites, and more than 60 percent of content-management systems. It's hard to imagine a bigger success.

But in the last few months, WordPress has become famous for something else, namely a whole lot of drama and lawsuits. It has certainly raised about open-source project governance, and the nature of profit and competition in the open-source world.

## Data and six questions
The data comes from two CSV files based on the cloning of the git repository wordpress-develop. One file with each of the commits in the trunk branch, and another with the number of lines added and removed in each commit. T

The first logfile was created with the following command
`git log --pretty=format:'%H~~%ad~~%an~~%ae~~%s' --date=iso > gitlog_basic.csv`

The above asks Git to produce a log using a format that I defined:
- The full 40-character commit ID, aka a SHA-1 hash
- The date of the commit
- The name of the committer
- The e-mail address of the committer
- A short version of the commit subject
Note that all of these are available using special % format codes, all documented in the manual for git log. I asked for the date to be formatted in ISO format, which I know that Pandas can handle directly. I then wrote this into a CSV file, gitlog_basic.csv.

But wait a second... what's with the \~~ characters? I used those to separate the fields on each line. I wanted to use commas, but there are commas in the Git subject lines, and getting the quotes to work was a bit more complex than I would have wanted. I tried my old standby, the tab character, but it turns out (!) that there are tabs in the subject lines, too! I decided to go with something that didn't actually show up in the Git log, namely ~~ 

## Challenges
The learning goals involve grouping, pivot tables, plotting, and joining.
- Import the wordpress-gitlog.csv file into a data frame. Make sure the date column has a datetime type. From the email column, create two new columns, email_user and email_domain, from the parts before and after the @ sign in the e-mail address. From the subject column, create a category column containing the category from before the first : character.
- Read the wordpress-numstats.csv file into a data frame. The three columns are the commit ID (SHA-1), the number of lines added in that commit, and the number of lines removed. Join the latter two columns into the main data frame.


Import the wordpress-gitlog.csv file into a data frame. Make sure the date column has a datetime type. From the email column, create two new columns, email_user and email_domain, from the parts before and after the @ sign in the e-mail address. From the subject column, create a category column containing the category from before the first : character.

In [22]:
import pandas as pd
filename = r"C:\Users\npigeon1\Pandas-Bamboo-Weekly-1\BW #102 WordPress\wordpress-gitlog.csv"

In [23]:
df = (pd
      .read_csv(filename, sep='~~', engine='python',
                names=['commit', 'date', 'name', 'email', 'subject'],
                parse_dates=['date'])
     )
df.dtypes

commit                  object
date       datetime64[ns, UTC]
name                    object
email                   object
subject                 object
dtype: object

In [24]:
df['email_uer']  = df['email'].str.extract(r'(.*)@')
df['email_domain'] = df['email'].str.extract(r'@(.*)')
df.shape
df.dtypes


commit                       object
date            datetime64[ns, UTC]
name                         object
email                        object
subject                      object
email_uer                    object
email_domain                 object
dtype: object

In [25]:
df1 = (pd
      .read_csv(filename, sep='~~', engine='python',
                names=['commit', 'date', 'name', 'email', 'subject'],
                parse_dates=['date'])
      .assign(category = lambda df_: df_['subject'].str.split(':').str.get(0),
              email_user = lambda df_: df_['email'].str.split('@').str.get(0),
              email_domain = lambda df_: df_['email'].str.split('@').str.get(-1))
     )
df1.shape
df1.dtypes

commit                       object
date            datetime64[ns, UTC]
name                         object
email                        object
subject                      object
category                     object
email_user                   object
email_domain                 object
dtype: object

50,596 rows and 8 columns.

Read the wordpress-numstats.csv file into a data frame. The three columns are the commit ID (SHA-1), the number of lines added in that commit, and the number of lines removed. Join the latter two columns into the main data frame.

In [28]:
filename2 = r'C:\Users\npigeon1\Pandas-Bamboo-Weekly-1\BW #102 WordPress\new-numstats.csv'
df2 = pd.read_csv(filename2,
                  names=['commit', 'added', 'deleted'],
                  sep='~~', engine='python')
df2.dtypes

commit     object
added       int64
deleted     int64
dtype: object

There are two ways to combine data frames in Pandas, with join and merge. Simply put, you use join when the two data frames share an index. You use merge when the data frames have common non-index columns. 

In [30]:
df_combined = df.merge(df2, on='commit')
df_combined.head()

,commit,date,name,email,subject,email_uer,email_domain,added,deleted
0,0f2334da8111913a2a78c5000f27f791bb405f0f,2025-01-21 22:57:04+00:00,Jb Audras,audrasjb@git.wordpress.org,"Formatting: Preserve `target=""_blank""` in Biog...",audrasjb,git.wordpress.org,76,2
1,eb50dd7cbf8bcbeda7521e1c152103d9d0c82009,2025-01-21 22:36:50+00:00,Jb Audras,audrasjb@git.wordpress.org,Customize: Show sidebar's description below it...,audrasjb,git.wordpress.org,4,1
2,61b7b9713ef6a71126c3739a929ce604762de1bf,2025-01-21 21:47:27+00:00,Jb Audras,audrasjb@git.wordpress.org,Themes: Remove title attributes from theme lis...,audrasjb,git.wordpress.org,9,12
3,396f6fbe4371affe13ba0eea4b88fb1e79d90902,2025-01-21 21:24:59+00:00,Weston Ruter,westonruter@git.wordpress.org,Menus: Improve performance by calling `get_pri...,westonruter,git.wordpress.org,44,39
4,f2f13cbff083e2dd7973565bdb9523f10746085e,2025-01-21 15:40:51+00:00,John Blackbourn,johnbillion@git.wordpress.org,Build/Test Tools: Switch to using local refere...,johnbillion,git.wordpress.org,68,38
